# 07 — RQ7: Answer metrics (RAGAS)

**Câu hỏi:** Đánh giá độ tin cậy (faithfulness) và tính liên quan (relevancy) theo chuẩn RAGAS.

**Metric:** Faithfulness, Answer Relevancy, Context Precision, Context Recall.

In [ ]:
# --- Setup ---
import sys
from pathlib import Path
import pandas as pd
import os

HERE = Path.cwd()
for p in [HERE] + list(HERE.parents):
    if (p / "source").is_dir() and (p / "research").is_dir():
        TRAFFIC_RAG = p
        break

sys.path.insert(0, str(TRAFFIC_RAG))
sys.path.insert(0, str(TRAFFIC_RAG.parent))

from research.utils.eval_runner import load_results, load_eval_set
RESULTS_DIR = TRAFFIC_RAG / "research" / "results" / "metrics"
EVAL_PATH = TRAFFIC_RAG / "research" / "data" / "eval_qa.jsonl"


## 1. Prepare Dataset for RAGAS

In [ ]:
from datasets import Dataset

# We use Agentic RAG results for this benchmark
agentic_results = load_results(RESULTS_DIR / "rq1_agentic_rag.jsonl")
gold_set = {r['id']: r for r in load_eval_set(EVAL_PATH)}

data_map = {
    "question": [],
    "answer": [],
    "contexts": [],
    "ground_truth": []
}

for r in agentic_results:
    gold = gold_set.get(r['id'])
    if not gold: continue
    
    data_map["question"].append(r['question'])
    data_map["answer"].append(r['answer'])
    # RAGAS expects list of strings for contexts
    data_map["contexts"].append(r.get('contexts', ["N/A"]))
    data_map["ground_truth"].append(gold['gold_answer'])

dataset = Dataset.from_dict(data_map)


## 2. Run Evaluation

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

result = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=llm,
)

print(result)
df_result = result.to_pandas()
df_result.to_csv(RESULTS_DIR / "rq7_ragas_detailed.csv", index=False)
